<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c35-header.png" alt="Nextia Learning · Forecasting and Time Series: Predict What Comes Next" width="100%">

# Deep and foundation models

**[Deep and foundation models](https://learning.nextia-ai.com/courses/forecasting/m05/deep-and-foundation-models/)** · Forecasting and Time Series: Predict What Comes Next · Module 5, lesson 3 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** run a pretrained time-series model (Chronos-Bolt small) on a CPU with no training, read its forecast as a range, add it to the backtest table, and measure what it costs: its size, its bias and the share of real values inside its range.

| | |
|---|---|
| **Time** | About 40 minutes, of which about 3 minutes of installs and downloads |
| **Needs** | An internet connection for the setup cells. No account and no key. The first cell also installs chronos-forecasting 2.3.2 and PyTorch 2.15.0 (CPU); section 1 downloads the model (about 190 MB). No GPU. |
| **You should know** | The backtest table, from [Measure the error that matters](https://learning.nextia-ai.com/courses/forecasting/m03/measure-the-error-that-matters/). LightGBM's row, from [Gradient boosting with LightGBM](https://learning.nextia-ai.com/courses/forecasting/m05/gradient-boosting-with-lightgbm/). |
| **Tested** | Python 3.12 with pandas 2.3.3, statsforecast 2.1.1, mlforecast 1.1.0, hierarchicalforecast 1.5.3, LightGBM 4.7.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*); chronos-forecasting 2.3.2 and PyTorch 2.15.0 (CPU) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m05/deep-and-foundation-models/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C35/M05-L03-deep-and-foundation-models/deep-and-foundation-models-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 2.3.3, statsforecast 2.1.1, chronos-forecasting 2.3.2, torch 2.15.0, and matplotlib for the charts. It needs an internet connection and takes about a minute the first time. PyTorch is large (about 200 MB to download): the first run of this cell can take 2 minutes.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==2.3.3 statsforecast==2.1.1 chronos-forecasting==2.3.2 torch==2.15.0 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
import pandas; print("pandas", pandas.__version__)
import statsforecast; print("statsforecast", statsforecast.__version__)
import torch; print("torch", torch.__version__)
import importlib.metadata as _md; print("chronos-forecasting", _md.version("chronos-forecasting"))

## Setup: get the practice data

Run the next cell. It downloads the Larkfield data of this course (about 2 MB) into a folder `larkfield-forecasting/data`, and the course's backtest helper `backtest.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: 403758 rows of daily sales (200 products, 3 regions), from 2024-04-01 to 2026-03-29.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-forecasting` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield data is synthetic: we made it for this course. It has no real people, products or prices.

In [ ]:
# Setup: download the course data into larkfield-forecasting/data/ and check it.
import gzip, hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C35/"
FILES = {
    "calendar.csv": "a5d1d5a3b640a983de72a30fca0e627a4ef23e797b1d80619e9ffcaadf118844",
    "demand.csv.gz": "8b39303985cbdb9bb1f3c705cd927ed6aaeb6f9665bd56ad4c3f8cb2b8578c99",
    "orders.csv": "bd8059fd73fe294daa157f8a6b6ae3f58319fa644eaa7ff0d31122b02cf05e30",
    "price_changes.csv": "889b258be4e933b87065d81d0771d6cd9b630ee90223b0a6ab074129c42153cb",
    "products.csv": "198b0a3de6eea8128af68f3cbc3a5c6acd3a64ac854fff18d37660857be8cfbf",
    "promotions.csv": "ca147959faa4c36c757742de7121976ffe79d59b53049dbbc8337a7547541f4a",
    "roster_hourly.csv": "6c2e0e96881b36ded52ed19458515c71f025288ddaebd589b50c82a8ad7fa69b",
    "tickets_daily.csv": "36f7067db060fdd2bd084c408c0622370a012773849f243020336f1fc678b31d",
    "tickets_hourly.csv.gz": "43f7fbb3e360a8e4a0cb4e101ea6a6242a76f0c37204d76ee9413ea3adbedd92",
    "weather.csv": "43a199a8b235940e291195bfd026901b62a10189b29f3a3ad9527b5382544276",
    "backtest.py": "764ecd38c6e96632be6f0e139e05eadd7b07d14a4ece73d881fc7559c2fa203a",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-forecasting").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and sha256(path) == expected:
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-forecasting and run again.")

for name, expected in FILES.items():
    if name == "backtest.py":
        fetch(name, LABS + name, PROJECT / name, expected)
    else:
        fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
with gzip.open("data/demand.csv.gz", "rt") as f:
    header = f.readline()
    rows = [line.split(",", 3)[:3] for line in f]
print(f"Ready: {len(rows)} rows of daily sales ({len({r[2] for r in rows})} products, {len({r[1] for r in rows})} regions), "
      f"from {min(r[0] for r in rows)} to {max(r[0] for r in rows)}.")

## Setup: helpers

The next cell loads pandas as `pd`, numpy as `np` and the course's backtest helper as `bt`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the backtest helper, and checks that do not show the answer.
import hashlib, json, logging, os, warnings
os.environ["NIXTLA_ID_AS_COL"] = "1"
import numpy as np
import pandas as pd
import backtest as bt
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")
logging.getLogger("lightning").setLevel(logging.ERROR)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def load_sales(products=None):
    """Daily sales per product, the three regions added up, in the long format of the backtest:
    unique_id (the product), ds (the day), y (units sold), price, on_promotion, stock_out.
    The missing day 2025-10-19 gets the mean of the same weekday one week before and after."""
    d = pd.read_csv("data/demand.csv.gz", parse_dates=["date"])
    if products is not None:
        d = d[d["product_id"].isin(products)]
    g = d.groupby(["product_id", "date"], as_index=False).agg(
        y=("sales", "sum"), price=("price", "mean"), on_promotion=("on_promotion", "max"), stock_out=("stock_out", "max"))
    out = []
    for uid, s in g.rename(columns={"product_id": "unique_id", "date": "ds"}).groupby("unique_id"):
        s = s.set_index("ds").reindex(pd.date_range(s["ds"].min(), s["ds"].max(), freq="D"))
        gap = s["y"].isna()
        before, after = s["y"].shift(7), s["y"].shift(-7)
        s.loc[gap, "y"] = ((before + after) / 2).fillna(before).fillna(after)[gap].round()
        s = s.ffill()
        s["unique_id"] = uid
        out.append(s.rename_axis("ds").reset_index())
    return pd.concat(out, ignore_index=True)[["unique_id", "ds", "y", "price", "on_promotion", "stock_out"]]

# The course backtest (the same in every module): 13 origins, 28 days apart, the last one 2026-03-15.
ORIGINS = bt.origins("2026-03-15", n=13, every="28D")
BACKTEST_PRODUCTS = ['P001', 'P002', 'P005', 'P006', 'P009', 'P013', 'P018', 'P024', 'P025', 'P029', 'P032', 'P039', 'P045', 'P057', 'P068', 'P075', 'P077', 'P078', 'P080', 'P081', 'P087', 'P088', 'P093', 'P095', 'P108', 'P110', 'P121', 'P130', 'P133', 'P136', 'P137', 'P141', 'P145', 'P147', 'P148', 'P156', 'P158', 'P159', 'P166', 'P169', 'P172', 'P174', 'P176', 'P183', 'P184', 'P186', 'P191', 'P192', 'P194', 'P199']
PRICES = pd.read_csv("data/products.csv").set_index("product_id")["base_price"].to_dict()

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Load a pretrained model

**Chronos-Bolt** is a model that its authors trained on many public time series. You do not train it on Larkfield: you give it a series, and it returns a forecast. The next cell downloads the small version at a fixed revision (about 190 MB, once), loads it on the CPU and counts its parameters: the numbers it learned.

> **Predict.** How many parameters has the small model: about 50 thousand, 50 million or 50 billion? Then run the cell.

In [ ]:
import time
import torch
from chronos import BaseChronosPipeline

t0 = time.perf_counter()
pipe = BaseChronosPipeline.from_pretrained("amazon/chronos-bolt-small", revision="772f3d25d38aec6d914c8949dab4462e2d46f5d8",
                                           device_map="cpu", torch_dtype=torch.float32)
print(f"loaded in {time.perf_counter() - t0:.1f} s")
params = round(sum(p.numel() for p in pipe.model.parameters()) / 1e6, 1)
print("parameters (millions):", params)

> **Check.** You should see `parameters (millions): 47.7`. The load time depends on your computer (a few seconds on a laptop, after the download). Warnings about Hugging Face (no token) or about redirects on macOS are harmless.

ETS keeps a handful of numbers per product. This model keeps 47.7 million, the same for every series it sees.

## 2. The parasol, with no training

The model reads only the numbers of the series: here the parasol's last 512 days up to Sunday 11 May 2025. It returns **quantiles**. The 0.1 quantile is a value that the real sales should fall below on about 1 day in 10; the 0.5 quantile (the median) is the point forecast; the 0.9 quantile is exceeded on about 1 day in 10.

> **Predict.** Will the median for the 14 days be above or below the real 216 parasols? Then run the cell.

In [ ]:
import matplotlib.pyplot as plt

parasol = load_sales(["P025"])
origin = pd.Timestamp("2025-05-11")
context = torch.tensor(parasol.loc[parasol["ds"] <= origin, "y"].to_numpy()[-512:], dtype=torch.float32)
q, _ = pipe.predict_quantiles([context], prediction_length=14, quantile_levels=[0.1, 0.5, 0.9])
fan = pd.DataFrame(q[0].numpy().clip(0), columns=["q10", "q50", "q90"])
fan["ds"] = pd.date_range(origin + pd.Timedelta(days=1), periods=14, freq="D")
fan = fan.merge(parasol[["ds", "y"]], on="ds")
inside = int(((fan["y"] >= fan["q10"]) & (fan["y"] <= fan["q90"])).sum())
print(fan[["q10", "q50", "q90"]].astype(float).sum().round(1).to_dict(), "| real:", int(fan["y"].sum()), "| days inside the range:", inside, "of 14")

before = parasol[(parasol["ds"] > origin - pd.Timedelta(days=28)) & (parasol["ds"] <= origin)]
plt.figure(figsize=(9, 3.5))
plt.plot(before["ds"], before["y"], color="grey", label="sales before the cut-off")
plt.fill_between(fan["ds"], fan["q10"], fan["q90"], alpha=0.25, label="0.1 to 0.9 quantile")
plt.plot(fan["ds"], fan["q50"], label="median forecast")
plt.plot(fan["ds"], fan["y"], "o", color="black", markersize=3, label="real sales")
plt.legend(loc="upper left"); plt.ylabel("parasols a day"); plt.show()

> **Check.** You should see a median of 181.8 for the 14 days against 216 real sales, a range from 96.7 to 309.7, and 12 of 14 days inside the range.

The median is too low here: the spring ramp grows faster than the model expects. The range is wide, and most real days fall inside it.

## 3. Chronos-Bolt in the course backtest

The next cell wraps the model as a forecaster for `backtest.py`, with the five quantiles of the course (0.05, 0.1, 0.5, 0.9, 0.95), and runs it for the 50 products at the 13 cut-offs: 650 forecasts with no training. Then it measures the **coverage** of the 80% range (0.1 to 0.9) and of the 90% range (0.05 to 0.95): the share of real values inside each.

> **Predict.** The 90% range should hold about 90% of the real values. Will it? Then run the cell (less than a minute on a laptop CPU).

In [ ]:
import logging
import chronos.chronos_bolt

LEVELS = [0.05, 0.1, 0.5, 0.9, 0.95]
pipe.predict_quantiles([context], prediction_length=14, quantile_levels=LEVELS)  # shows Chronos's warning once
logging.getLogger(chronos.chronos_bolt.__file__).setLevel(logging.ERROR)       # then hides it for the backtest

def chronos(train, h, future):
    t = train.sort_values(["unique_id", "ds"])
    ids = sorted(t["unique_id"].unique())
    ctx = [torch.tensor(g["y"].to_numpy()[-512:], dtype=torch.float32) for _, g in t.groupby("unique_id", sort=True)]
    q, _ = pipe.predict_quantiles(ctx, prediction_length=h, quantile_levels=LEVELS)
    q = q.numpy().clip(0)
    out = []
    for k, uid in enumerate(ids):
        days = future.loc[future["unique_id"] == uid, "ds"].sort_values().to_numpy()[:h]
        block = pd.DataFrame({"unique_id": uid, "ds": days})
        for j, lv in enumerate(LEVELS):
            block[f"q{int(round(lv * 100))}"] = q[k, :len(days), j]
        out.append(block)
    out = pd.concat(out, ignore_index=True)
    return out.assign(yhat=out["q50"])

sales = load_sales(BACKTEST_PRODUCTS)
fc_chronos = bt.backtest(sales, chronos, h=14, origins=ORIGINS)
row = bt.score(fc_chronos, sales, "Chronos-Bolt small, no training", season=7, cost=bt.stock_cost(PRICES))
table = bt.show([row])
print("coverage of the 80% range:", bt.coverage(fc_chronos, "q10", "q90"))
print("coverage of the 90% range:", bt.coverage(fc_chronos, "q5", "q95"))

> **Check.** First you see Chronos's own warning: the quantiles 0.05 and 0.95 are not among the quantiles it was trained on, so it uses its 0.1 and 0.9 instead. Then you see MASE 0.784, WAPE % 38.7, bias % -5.6, pinball 0.099 and a cost of 14,750 dollars, then a coverage of 0.789 for the 80% range and 0.789 for the 90% range.

The best MASE and WAPE of the backtest table, with no training at all. But look at the other numbers. The bias is negative: the forecasts are too low, by 5.6% in total. And the "90% range" holds the same share as the 80% range: the model returns only the quantiles 0.1 to 0.9, so it cannot give a wider range, whatever you ask for.

## 4. Your turn: what the model does not see

The model reads only the units sold. It does not know the promotion calendar, which Larkfield plans in advance.

> **Your turn.** In `fc_chronos`, keep the rows where the product was on promotion (join `sales[["unique_id", "ds", "on_promotion"]]` on `unique_id` and `ds`). Compute the bias in % of those rows: `100 * (forecast sum - real sum) / real sum`, rounded to 1 decimal. Put it in `promo_bias_chronos` as a Python float. Then run the check cell.

In [ ]:
promo_bias_chronos = ...  # the bias in % of Chronos-Bolt on the promotion days

In [ ]:
expect('promo_bias_chronos', promo_bias_chronos, -45.0)

On the 154 promotion days, Chronos-Bolt misses almost half of the units, as ETS does (−43.5%). LightGBM, which sees the promotion flag, was a few percent too high. Some versions of Chronos accept known future inputs such as a promotion flag; the version of this course does not use them.

## Next

You ran a foundation model with no training and measured its accuracy and what it costs. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m05/deep-and-foundation-models/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [When not to build a model](https://learning.nextia-ai.com/courses/forecasting/m05/when-not-to-build-a-model/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/forecasting/m05/deep-and-foundation-models/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.